## Resume Analysis

In [3]:
import sys
print(sys.executable)

/home/sumitha_m/Datheon_Python_Generative_AI_LangChain_Internship/Datheon_Generative_AI_Internship/.venv/bin/python


In [2]:
import  os, json, pdfplumber
from dotenv import load_dotenv
from openai import OpenAI
from base64 import b64encode
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from pathlib import Path

load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
MODEL="openai/gpt-oss-20b"
DATA = "/home/sumitha_m/Datheon_Python_Generative_AI_LangChain_Internship/Datheon_Generative_AI_Internship/data"

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

with open("data/resumes/03_daniel_okafor_banner_header_chips.pdf", "rb") as f:
    document = b64encode(f.read()).decode()

In [13]:
def read_pdf(path):
    with pdfplumber.open(path) as pdf:
        return "\n".join(page.extract_text() or "" for page in pdf.pages)
pdf_path = Path("/home/sumitha_m/Documents/Job_search/Sumitha_Manoharan_Resume_GenAI_Developer.pdf")        
# pdf_path = Path(f"{DATA}{DATA}/resumes/02_emily_vargas_two_column_sidebar.pdf")  
resume_text = read_pdf(pdf_path)
print(resume_text[:500])

SUMITHA MANOHARAN
Coimbatore, India | sumithamanoharan76@gmail.com | +91 9994170602 | linkedin.com/in/sumitha-manoharan | Kestrel Demo
PROFESSIONAL SUMMARY
Generative AI developer with certified training in Data Science and Machine Learning and hands-on experience building
LLM applications: Kestrel, a RAG advisor over research papers (text-embedding-3-small, Chroma, GPT-OSS 120B) deployed
on Hugging Face with an offline evaluation harness, and a LangChain code review agent. Strong Python foundat


In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")

class Experience(Strict):
    company : str
    job_title: str
    job_location: str
    start_date: str | None = Field(description="YYYY-MM, or null if not stated")
    end_date: str | None = Field(description="YYYY-MM, or null if the job is current")
    is_current: bool
    highlights: list[str] = Field(
        description="Key responsibilities and achievements for this job, one bullet per entry. "
                    "Stay close to the resume's wording and do not invent anything. "
                    "Empty list if the resume gives no details."
    )

class Education(Strict):
    institution: str | None
    degree: str | None = Field(description="example: B.Sc,M.Sc, MS, BE etc")
    field_of_study: str | None = Field(description=" actual field they got their masters or UG in example: maths, robotics, datascience and analytics etc ")
    start_year: str | None = Field(description="YYYY only. Null if not stated.")
    end_year: str | None = Field(description="YYYY only. Null if not stated.")
    grade: str | None = Field(description="Grade or GPA exactly as written, null if not stated")

class Projects(Strict):
    project_title: str
    project_description: str
    skills_used: list[str] | None = Field(description="skills or technologies used in the project")


class Resume(Strict):
    is_resume: bool = Field(description="true if its a resume, false if anything other than a resume")    
    language: str = Field(description="language of the resume")
    name: str
    headline: str
    email: str | None
    phone : str | None
    location: str | None
    skills: list[str] = Field(description="One skill per entry")
    experience: list[Experience]
    education: list[Education]
    certifications: list[str] = Field(description="certifications of the candidate")
    projects: list[Projects] | None 


    


In [6]:
def extract(model, text):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You extract data from resumes. Use only what is written. "
                                          "If something is missing use null. Write dates as YYYY-MM."},
            {"role": "user", "content": f"Resume:\n\n{text}"},
        ],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "resume", "strict": True, "schema": Resume.model_json_schema()}},
        extra_body={"provider": {"require_parameters": True}},
    )
    raw = response.choices[0].message.content
    return Resume.model_validate_json(raw)      # the format check: raises ValidationError if wrong

In [15]:
result = extract(MODEL, resume_text)
print(result.model_dump_json(indent=2))

{
  "is_resume": true,
  "language": "en",
  "name": "Sumitha Manoharan",
  "headline": "Generative AI developer",
  "email": "sumithamanoharan76@gmail.com",
  "phone": "+91 9994170602",
  "location": "Coimbatore, India",
  "skills": [
    "Generative AI & LLMs: RAG, embeddings & vector databases (OpenAI embeddings, Chroma), prompt design, LLM‑as‑judge evaluation, LLM tool‑calling (LangChain & native), LLM API integration (LiteLLM, OpenRouter)",
    "AI / ML Foundations: Machine learning (Imarticus certification – advanced ML track), NLP (spaCy, TF‑IDF), Pandas, data analysis",
    "Programming & Data: Python, SQL (basic – SELECT, JOIN, aggregation), JSON, Pydantic (structured LLM output), structured & unstructured data",
    "APIs & Backend: REST API integration, FastAPI, Flask (Udemy Python Pro Bootcamp), Gradio",
    "Tools & Cloud: Git/GitHub, Hugging Face Spaces (deployment), AWS S3, JIRA"
  ],
  "experience": [
    {
      "company": "Dataweave Private Limited",
      "job_title"

In [ ]:
# import base64, mimetypes, json, os
# from dotenv import load_dotenv
# from openai import OpenAI

# load_dotenv(override=True)
# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))
# MODEL = "scaledown/extract"

# ENTITIES = {
#     "is_resume": {"labels": [
#         {"name": "yes", "rubric": "the document is a resume or CV"},
#         {"name": "no", "rubric": "the document is anything else"},
#     ]},
#     "full_name": "Find the full name of the candidate",
#     "headline": "Find the professional headline or job title under the name",
#     "email": "Find the candidate's email address",
#     "phone": "Find the candidate's phone number",
#     "location": "Find the candidate's city or location",
#     "link": "Find a URL such as LinkedIn, GitHub or a portfolio",
#     "skill": "Find a single technical or professional skill",
#     "company": "Find the name of an employer",
#     "job title": "Find a job title held at an employer",
#     "institution": "Find the name of a school or university",
#     "degree": "Find a degree obtained",
# }

# def file_part(path):
#     mime = mimetypes.guess_type(path)[0]
#     with open(path, "rb") as f:
#         data_url = f"data:{mime};base64," + base64.b64encode(f.read()).decode()
#     if mime == "application/pdf":
#         return {"type": "file", "file": {"filename": os.path.basename(path), "file_data": data_url}}
#     return {"type": "image_url", "image_url": {"url": data_url}}

# path = "data/resumes/08_invoice_not_a_resume_invoice.pdf"

# response = client.chat.completions.create(
#     model=MODEL,
#     messages=[{"role": "user", "content": [
#         {"type": "text", "text": "Extract the resume fields."},
#         file_part(path),
#     ]}],
#     extra_body={"provider": {"options": {"scaledown": {"entities": ENTITIES}}}},
# )

# result = json.loads(response.choices[0].message.content)
# print(result.keys())
# print(result["entities"])
    # print(f'{e["type"]:12} | {e["text"]} | {e["confidence"]:.2f}')

dict_keys(['entities', 'structured_result', 'ocr_text', 'input_tokens'])
[{'text': 'billing@brightline-supplies.example.com', 'type': 'email', 'confidence': 0.998, 'start': 0, 'end': 0, 'context': 'Extract the resume fields.'}, {'text': 'Springfield, 00000', 'type': 'location', 'confidence': 0.841, 'start': 0, 'end': 0, 'context': 'Extract the resume fields.'}, {'text': 'Riverton, 00001', 'type': 'location', 'confidence': 0.997, 'start': 0, 'end': 0, 'context': 'Extract the resume fields.'}, {'text': 'Brightline Office Supplies Ltd.', 'type': 'company', 'confidence': 0.999, 'start': 0, 'end': 0, 'context': 'Extract the resume fields.'}, {'text': 'Harlow & Finch Architects', 'type': 'company', 'confidence': 0.998, 'start': 0, 'end': 0, 'context': 'Extract the resume fields.'}, {'text': 'no', 'type': 'is_resume', 'confidence': 0.9875683484920422, 'start': 0, 'end': 0, 'context': 'Extract the resume fields.'}]
